<a href="https://colab.research.google.com/github/rafaalathais/analise-gasto-parlamentar/blob/main/notebook_gasto_parlamentar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gastos da CEAPS dos senadores
**Disciplina**: IMD1151 - Ciências de Dados

**Grupo**:
*   Gabriel Ferreira Cavalcante
*   Jordson Albino Galvão Tavares
*   Rafaela Thais de Abreu Lúcio Maia

## Introdução

Este projeto tem como objetivo realizar uma Análise Exploratória de Dados sobre os gastos da Cota para o Exercício da Atividade Parlamentar dos Senadores (CEAPS), buscando identificar padrões, tendências e diferenças na utilização dos recursos públicos destinados à atividade parlamentar.

Nesta etapa, analisamos os anos completos de 2008 a 2025. Os dados de 2026 ficam para uma análise separada.

### Objetivo

Serão realizadas a obtenção dos registros históricos, a identificação das variáveis disponíveis e a verificação inicial de problemas como dados ausentes, inconsistências e diferenças de estrutura entre os arquivos.

Os dados consolidados servirão de base para as etapas posteriores de limpeza, transformação e análise exploratória.

## Perguntas de pesquisa

O projeto busca responder às seguintes perguntas:

- Em quais períodos do ano os senadores apresentam maiores gastos? Como essas despesas evoluíram ao longo dos anos?

- Quais unidades federativas apresentam os maiores gastos associados aos seus senadores?

- Quais partidos apresentam os maiores gastos totais e médios, tanto nacionalmente quanto por unidade federativa?

- Quais senadores apresentam os maiores gastos da cota parlamentar?

- Quais tipos de despesas representam a maior parcela dos gastos parlamentares?

Como possível extensão do projeto, pretende-se investigar a identificação de perfis de senadores com base em seus padrões de despesas.

## Fontes de dados

Os dados serão obtidos por meio do Portal de Dados Abertos do Senado Federal.


## Coleta da CEAPS

A CEAPS está no [Portal de Dados Abertos do Senado](https://www12.senado.leg.br/dados-abertos/administrativo/senadores/cotas-para-exercicio-da-atividade-parlamentar). A API administrativa oferece um CSV por ano em `https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores/despesas_ceaps/{ano}/csv`.

Os CSVs de 2008 a 2025 ficam, sem alterações, em `dados_cache/`. O notebook reutiliza esses arquivos nas próximas execuções. A tabela de coleta mostra o resultado de cada ano. Se algum ano falhar, o notebook interrompe a preparação. Assim, uma coleta parcial não é tratada como completa.

In [3]:
from io import BytesIO
from pathlib import Path
import time
import unicodedata

import pandas as pd
import requests
from IPython.display import Markdown, display

ANOS = list(range(2008, 2026))
URL_CEAPS = 'https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores/despesas_ceaps/{}/csv'
CACHE = Path('dados_cache')
CACHE.mkdir(exist_ok=True)

def obter_csv(ano):
    caminho = CACHE / f'ceaps_{ano}.csv'
    if caminho.is_file() and caminho.stat().st_size > 0:
        return caminho.read_bytes(), 'cache'
    ultimo_erro = None
    for tentativa in range(3):
        try:
            resposta = requests.get(URL_CEAPS.format(ano), timeout=(15, 90))
            resposta.raise_for_status()
            if not resposta.content:
                raise ValueError('resposta vazia')
            return resposta.content, 'download'
        except (requests.RequestException, ValueError) as erro:
            ultimo_erro = erro
            if tentativa < 2:
                time.sleep(2 ** tentativa)
    raise RuntimeError(f'ano {ano}: {ultimo_erro}')


In [4]:
dfs_despesas = {}
estado_coleta = []
colunas_essenciais = {'ID', 'ANO', 'MÊS', 'COD_SENADOR', 'VALOR_REEMBOLSADO'}

for ano in ANOS:
    try:
        conteudo, origem = obter_csv(ano)
        quadro = pd.read_csv(BytesIO(conteudo), sep=';', dtype='string', encoding='utf-8-sig')
        if quadro.empty or not colunas_essenciais.issubset(quadro.columns):
            raise ValueError('CSV vazio ou sem as colunas essenciais')
        dfs_despesas[ano] = quadro
        if origem == 'download':
            (CACHE / f'ceaps_{ano}.csv').write_bytes(conteudo)
        estado_coleta.append({'ano': ano, 'origem': origem, 'registros': len(quadro), 'erro': ''})
    except Exception as erro:
        estado_coleta.append({'ano': ano, 'origem': 'falha', 'registros': 0, 'erro': str(erro)})

tabela_coleta = pd.DataFrame(estado_coleta)
display(tabela_coleta)
anos_falhos = tabela_coleta.loc[tabela_coleta.origem.eq('falha'), 'ano'].tolist()
if anos_falhos:
    raise RuntimeError(f'Coleta CEAPS incompleta. Anos com falha: {anos_falhos}.')


,ano,origem,registros,erro
0,2008,cache,2449,
1,2009,cache,16480,
2,2010,cache,18235,
3,2011,cache,25574,
4,2012,cache,30175,
5,2013,cache,29376,
6,2014,cache,22002,
7,2015,cache,26055,
8,2016,cache,26768,
9,2017,cache,27023,


In [5]:
display(dfs_despesas[2025].head(5))


,ID,TIPO_DOCUMENTO,ANO,MÊS,COD_SENADOR,NOME_SENADOR,TIPO_DESPESA,CPF_CNPJ_FORNECEDOR,NOME_FORNECEDOR,DOCUMENTO,DATA,DETALHAMENTO,VALOR_REEMBOLSADO
0,2250668,Nota Fiscal Eletrônica,2025,2,5672,ALAN RICK,"Locomoção, hospedagem, alimentação, combustíve...",06.906.483/0001-74,POSTO VILLAGE LTDA,73594,2025-03-06,<NA>,"5130,07"
1,2249896,Nota Fiscal Eletrônica,2025,2,5672,ALAN RICK,"Locomoção, hospedagem, alimentação, combustíve...",07.454.351/0001-11,QUILIMANJARO COMDE ALIMHOTELLTDA,179158,2025-02-22,<NA>,"83,80"
2,2253644,Fatura,2025,3,5672,ALAN RICK,"Aluguel de imóveis para escritório político, c...",66.970.229/0132-26,CLARO NXT TELECOMUNICAÇÕES S.A,733127,2025-03-18,<NA>,"418,01"
3,2253646,Recibo,2025,3,5672,ALAN RICK,"Aluguel de imóveis para escritório político, c...",860.***.***-53,LARISSA MAIA MARTINELLO,03/2025,2025-03-25,<NA>,"4000,00"
4,2252906,Fatura,2025,3,5672,ALAN RICK,"Contratação de consultorias, assessorias, pesq...",05.120.923/0001-09,AEROTUR SERVIÇOS,2223234094,2025-03-11,<NA>,"1469,95"


In [6]:
colunas_referencia = set(dfs_despesas[ANOS[0]].columns)
tabela_estrutura = pd.DataFrame([{
    'ano': ano,
    'colunas': len(quadro.columns),
    'ausentes_vs_2008': ', '.join(sorted(colunas_referencia - set(quadro.columns))),
    'adicionais_vs_2008': ', '.join(sorted(set(quadro.columns) - colunas_referencia))
} for ano, quadro in sorted(dfs_despesas.items())])

df_ceaps_bruto = pd.concat(
    [dfs_despesas[ano].assign(ANO_ARQUIVO=ano) for ano in ANOS],
    ignore_index=True, sort=False
)
assert len(df_ceaps_bruto) == tabela_coleta.registros.sum()
print(f'CEAPS bruta: {len(df_ceaps_bruto):,} registros, de {ANOS[0]} a {ANOS[-1]}.')
display(tabela_estrutura)


CEAPS bruta: 382,061 registros, de 2008 a 2025.


,ano,colunas,ausentes_vs_2008,adicionais_vs_2008
0,2008,13,,
1,2009,13,,
2,2010,13,,
3,2011,13,,
4,2012,13,,
5,2013,13,,
6,2014,13,,
7,2015,13,,
8,2016,13,,
9,2017,13,,


## Dicionário de dados

O dicionário usa a base coletada. Valores únicos indica quantos valores diferentes existem na base inteira. Não somamos as contagens anuais, pois um mesmo valor pode aparecer em vários anos. Mantemos os identificadores e os campos originais. `DATA` também fica na base para uma possível integração por data.

| Campo | Interpretação | Uso nesta etapa |
|---|---|---|
| `ID` | Identificador do registro CEAPS | Preservado e verificado para repetições |
| `ANO`, `MÊS` | Ano e mês de competência informados pela CEAPS | Convertidos e validados |
| `COD_SENADOR` | Código do parlamentar na CEAPS | Preservado como texto para integração |
| `NOME_SENADOR` | Nome informado no registro | Espaços tratados; chave e nome de exibição derivados |
| `TIPO_DESPESA` | Categoria informada | Espaços tratados; rótulo padronizado derivado |
| `CPF_CNPJ_FORNECEDOR` | Documento do fornecedor | Preservado como texto, inclusive zeros à esquerda |
| `NOME_FORNECEDOR` | Nome informado do fornecedor | Espaços tratados; chave de comparação derivada |
| `VALOR_REEMBOLSADO` | Valor nominal em reais | Convertido para número, sem eliminar negativos |
| `DATA` | Data do comprovante, quando presente | Data ISO convertida em coluna separada |
| `TIPO_DOCUMENTO`, `DOCUMENTO`, `DETALHAMENTO` | Campos de comprovante e texto livre | Mantidos; ausências documentadas |
| `ANO_ARQUIVO` | Ano do CSV de origem | Criado para auditoria da coleta |

Na limpeza, criamos `NOME_SENADOR_CHAVE`, `NOME_FORNECEDOR_CHAVE` e `TIPO_DESPESA_CHAVE` para comparar textos sem depender de maiúsculas ou acentos. `NOME_SENADOR_PADRONIZADO` e `TIPO_DESPESA_PADRONIZADO` usam grafias que já existem na fonte. `DATA_COMPROVANTE` guarda a data convertida. As tabelas abaixo mostram os tipos, os valores ausentes e as contagens.

In [7]:
resumo_dicionario_bruto = pd.DataFrame([{
    'coluna': coluna,
    'tipo bruto': str(df_ceaps_bruto[coluna].dtype),
    'valores únicos': int(df_ceaps_bruto[coluna].nunique(dropna=True)),
    'ausentes': int(df_ceaps_bruto[coluna].isna().sum()),
    'ausentes (%)': round(df_ceaps_bruto[coluna].isna().mean() * 100, 2)
} for coluna in df_ceaps_bruto.columns])
display(resumo_dicionario_bruto)


,coluna,tipo bruto,valores únicos,ausentes,ausentes (%)
0,ID,string,382061,0,0.00
1,TIPO_DOCUMENTO,string,9,3144,0.82
2,ANO,string,18,0,0.00
3,MÊS,string,12,0,0.00
4,COD_SENADOR,string,326,0,0.00
5,NOME_SENADOR,string,326,0,0.00
6,TIPO_DESPESA,string,7,284,0.07
7,CPF_CNPJ_FORNECEDOR,string,30734,3144,0.82
8,NOME_FORNECEDOR,string,59795,3144,0.82
9,DOCUMENTO,string,261811,27534,7.21


## Diagnóstico da fonte

O notebook só reúne os CSVs anuais quando eles têm as colunas essenciais. A tabela de estrutura compara as colunas de cada ano com as de 2008. Há valores ausentes em campos de texto livre e de comprovante. Por isso, `DETALHAMENTO` não descreve todas as despesas. As contagens e os percentuais são calculados a cada execução, pois a API pode atualizar arquivos antigos.

## Limpeza da CEAPS

Guardamos os dados originais no DataFrame `df_ceaps_bruto`. O DataFrame `df_ceaps_limpo` mantém todas as linhas e os identificadores. Criamos colunas para comparar textos sem substituir as grafias originais. Não removemos automaticamente valores ausentes, negativos, outliers ou possíveis duplicatas. Negativos podem ser ajustes contábeis, e registros parecidos podem representar documentos diferentes.

In [8]:
df_ceaps_limpo = df_ceaps_bruto.copy()

def texto_basico(serie):
    return serie.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True).replace('', pd.NA)

def chave_texto(serie):
    return serie.map(lambda valor: ''.join(
        letra for letra in unicodedata.normalize('NFKD', valor.casefold())
        if not unicodedata.combining(letra)
    ) if pd.notna(valor) else pd.NA)

for coluna in ['NOME_SENADOR', 'NOME_FORNECEDOR', 'TIPO_DESPESA', 'CPF_CNPJ_FORNECEDOR']:
    df_ceaps_limpo[coluna] = texto_basico(df_ceaps_limpo[coluna])

df_ceaps_limpo['COD_SENADOR'] = texto_basico(df_ceaps_limpo['COD_SENADOR'])
df_ceaps_limpo['ID'] = texto_basico(df_ceaps_limpo['ID'])
for coluna in ['NOME_SENADOR', 'NOME_FORNECEDOR', 'TIPO_DESPESA']:
    df_ceaps_limpo[coluna + '_CHAVE'] = chave_texto(df_ceaps_limpo[coluna])

nomes = df_ceaps_limpo.groupby(['COD_SENADOR', 'NOME_SENADOR'], dropna=True).size().rename('n').reset_index()
nomes = nomes.sort_values(['n', 'NOME_SENADOR'], ascending=[False, True]).drop_duplicates('COD_SENADOR')
df_ceaps_limpo['NOME_SENADOR_PADRONIZADO'] = df_ceaps_limpo.COD_SENADOR.map(nomes.set_index('COD_SENADOR').NOME_SENADOR)

categorias = df_ceaps_limpo.groupby(['TIPO_DESPESA_CHAVE', 'TIPO_DESPESA'], dropna=True).size().rename('n').reset_index()
categorias = categorias.sort_values(['n', 'TIPO_DESPESA'], ascending=[False, True]).drop_duplicates('TIPO_DESPESA_CHAVE')
df_ceaps_limpo['TIPO_DESPESA_PADRONIZADO'] = df_ceaps_limpo.TIPO_DESPESA_CHAVE.map(
    categorias.set_index('TIPO_DESPESA_CHAVE').TIPO_DESPESA)

df_ceaps_limpo['ANO'] = pd.to_numeric(df_ceaps_limpo.ANO, errors='coerce').astype('Int64')
df_ceaps_limpo['MÊS'] = pd.to_numeric(df_ceaps_limpo['MÊS'], errors='coerce').astype('Int64')
valor_texto = df_ceaps_limpo.VALOR_REEMBOLSADO.astype('string').str.strip()
com_virgula = valor_texto.str.contains(',', regex=False, na=False)
valor_texto = valor_texto.where(~com_virgula, valor_texto.str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
df_ceaps_limpo['VALOR_REEMBOLSADO'] = pd.to_numeric(valor_texto, errors='coerce')
df_ceaps_limpo['DATA_COMPROVANTE'] = pd.to_datetime(df_ceaps_limpo.DATA, format='%Y-%m-%d', errors='coerce')

assert len(df_ceaps_limpo) == len(df_ceaps_bruto)
assert df_ceaps_limpo.ID.fillna('').equals(df_ceaps_bruto.ID.fillna(''))
assert df_ceaps_limpo.COD_SENADOR.fillna('').equals(df_ceaps_bruto.COD_SENADOR.fillna(''))
print(f'Base limpa: {len(df_ceaps_limpo):,} registros; nenhum registro removido.')


Base limpa: 382,061 registros; nenhum registro removido.


In [9]:
datas_validas = df_ceaps_limpo.loc[df_ceaps_limpo['DATA_COMPROVANTE'].notna(),
    ['ID', 'DATA', 'DATA_COMPROVANTE', 'ANO', 'MÊS']]

print("10 menores datas:")
print(datas_validas.nsmallest(10, 'DATA_COMPROVANTE').to_string(index=False))
print("\n10 maiores datas:")
print(datas_validas.nlargest(10, 'DATA_COMPROVANTE').to_string(index=False))

10 menores datas:
     ID       DATA DATA_COMPROVANTE  ANO  MÊS
2217253 0023-10-07       0023-10-07 2023   12
2217248 0023-10-27       0023-10-27 2023   12
2217246 0023-10-27       0023-10-27 2023   12
2051549 0200-10-30       0200-10-30 2016   10
2060535 0201-03-01       0201-03-01 2017    3
2016913 0201-08-25       0201-08-25 2015    8
2084537 0201-11-08       0201-11-08 2017   11
2249784 0202-02-24       0202-02-24 2025    2
2144535 0202-05-04       0202-05-04 2020    5
2233647 0202-07-04       0202-07-04 2024    7

10 maiores datas:
     ID       DATA DATA_COMPROVANTE  ANO  MÊS
2007486 5201-05-08       5201-05-08 2015    5
2050224 5017-07-20       5017-07-20 2016   10
2011103 5015-02-24       5015-02-24 2015    4
2034257 3016-04-02       3016-04-02 2016    4
2029364 2915-10-06       2915-10-06 2015   10
2076233 2107-08-29       2107-08-29 2017    9
2045115 2106-07-20       2106-07-20 2016    7
2039681 2106-05-03       2106-05-03 2016    5
2035589 2106-04-13       2106-04-13 2016   

In [10]:
ano_invalido = ~df_ceaps_limpo.ANO.between(2008, 2025).fillna(False)
ano_divergente = ~df_ceaps_limpo.ANO.eq(df_ceaps_limpo.ANO_ARQUIVO).fillna(False)
mes_invalido = ~df_ceaps_limpo['MÊS'].between(1, 12).fillna(False)
data_invalida = df_ceaps_bruto.DATA.notna() & df_ceaps_limpo.DATA_COMPROVANTE.isna()
valor_invalido = df_ceaps_bruto.VALOR_REEMBOLSADO.notna() & df_ceaps_limpo.VALOR_REEMBOLSADO.isna()
documento_digitos = df_ceaps_limpo.CPF_CNPJ_FORNECEDOR.str.replace(r'\D', '', regex=True)
documento_mascarado = df_ceaps_limpo.CPF_CNPJ_FORNECEDOR.str.contains('*', regex=False, na=False)
documento_formato_incomum = df_ceaps_limpo.CPF_CNPJ_FORNECEDOR.notna() & ~documento_mascarado & ~documento_digitos.str.len().isin([11, 14]).fillna(False)

valores = df_ceaps_limpo.VALOR_REEMBOLSADO.dropna()
q1, q3 = valores.quantile([.25, .75])
limite_inferior = q1 - 1.5 * (q3 - q1)
limite_superior = q3 + 1.5 * (q3 - q1)
diagnostico = pd.DataFrame([
    ['ANO fora do recorte', int(ano_invalido.sum()), 'mantido; revisar antes de agrupar por ano'],
    ['ANO diferente do arquivo', int(ano_divergente.sum()), 'mantido; revisar origem'],
    ['MÊS inválido', int(mes_invalido.sum()), 'mantido; não usar em agrupamento mensal'],
    ['DATA preenchida mas inválida', int(data_invalida.sum()), 'texto original mantido'],
    ['VALOR preenchido mas inválido', int(valor_invalido.sum()), 'texto original mantido na base bruta'],
    ['CPF parcialmente mascarado na fonte', int(documento_mascarado.sum()), 'mantido; não reconstruir dígitos ocultos'],
    ['CPF/CNPJ não mascarado com comprimento incomum', int(documento_formato_incomum.sum()), 'mantido; revisar sem presumir dígitos'],
    ['VALOR negativo', int(valores.lt(0).sum()), 'mantido como possível ajuste contábil'],
    ['VALOR extremo (regra 1,5 × IQR)', int((valores.lt(limite_inferior) | valores.gt(limite_superior)).sum()), 'mantido; regra apenas sinaliza revisão'],
], columns=['verificação', 'registros afetados', 'decisão'])
display(diagnostico)


,verificação,registros afetados,decisão
0,ANO fora do recorte,0,mantido; revisar antes de agrupar por ano
1,ANO diferente do arquivo,0,mantido; revisar origem
2,MÊS inválido,0,mantido; não usar em agrupamento mensal
3,DATA preenchida mas inválida,0,texto original mantido
4,VALOR preenchido mas inválido,0,texto original mantido na base bruta
5,CPF parcialmente mascarado na fonte,8662,mantido; não reconstruir dígitos ocultos
6,CPF/CNPJ não mascarado com comprimento incomum,60,mantido; revisar sem presumir dígitos
7,VALOR negativo,219,mantido como possível ajuste contábil
8,"VALOR extremo (regra 1,5 × IQR)",39553,mantido; regra apenas sinaliza revisão


In [11]:
colunas_originais = list(df_ceaps_bruto.columns)
tipos_e_ausencias = pd.DataFrame([{
    'coluna': coluna,
    'tipo antes': str(df_ceaps_bruto[coluna].dtype),
    'tipo depois': str(df_ceaps_limpo[coluna].dtype),
    'ausentes antes': int(df_ceaps_bruto[coluna].isna().sum()),
    'ausentes depois': int(df_ceaps_limpo[coluna].isna().sum()),
    'valores únicos na base bruta': int(df_ceaps_bruto[coluna].nunique(dropna=True))
} for coluna in colunas_originais])
display(tipos_e_ausencias)
colunas_derivadas = [coluna for coluna in df_ceaps_limpo.columns if coluna not in colunas_originais]
dicionario_derivado = pd.DataFrame([{
    'coluna derivada': coluna,
    'tipo': str(df_ceaps_limpo[coluna].dtype),
    'ausentes': int(df_ceaps_limpo[coluna].isna().sum())
} for coluna in colunas_derivadas])
display(dicionario_derivado)

comparacao = pd.DataFrame([
    ['registros', len(df_ceaps_bruto), len(df_ceaps_limpo)],
    ['IDs em repetição', int(df_ceaps_bruto.ID.duplicated(keep=False).sum()), int(df_ceaps_limpo.ID.duplicated(keep=False).sum())],
    ['linhas originais idênticas adicionais', int(df_ceaps_bruto.duplicated().sum()), int(df_ceaps_limpo[colunas_originais].duplicated().sum())],
], columns=['métrica', 'antes', 'depois'])
display(comparacao)

alteracoes = pd.DataFrame([{
    'campo': coluna,
    'registros alterados': int(df_ceaps_bruto[coluna].fillna('').ne(df_ceaps_limpo[coluna].fillna('')).sum())
} for coluna in ['NOME_SENADOR', 'NOME_FORNECEDOR', 'TIPO_DESPESA', 'CPF_CNPJ_FORNECEDOR']])
alteracoes.loc[len(alteracoes)] = ['rótulo de categoria padronizado por grafia existente',
    int(df_ceaps_limpo.TIPO_DESPESA.fillna('').ne(df_ceaps_limpo.TIPO_DESPESA_PADRONIZADO.fillna('')).sum())]
alteracoes.loc[len(alteracoes)] = ['nome de senador de exibição por COD_SENADOR',
    int(df_ceaps_limpo.NOME_SENADOR.fillna('').ne(df_ceaps_limpo.NOME_SENADOR_PADRONIZADO.fillna('')).sum())]
display(alteracoes)

def converter_valor(serie):
    texto = serie.astype('string').str.strip()
    com_virgula = texto.str.contains(',', regex=False, na=False)
    texto = texto.where(~com_virgula, texto.str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    return pd.to_numeric(texto, errors='coerce')

total_bruto = converter_valor(df_ceaps_bruto.VALOR_REEMBOLSADO).sum()
total_limpo = df_ceaps_limpo.VALOR_REEMBOLSADO.sum()
print(f'Total antes: R$ {total_bruto:,.2f}; depois: R$ {total_limpo:,.2f}; diferença: R$ {total_limpo-total_bruto:,.2f}.')
assert len(df_ceaps_limpo) == len(df_ceaps_bruto)
assert abs(total_limpo - total_bruto) < .01
assert df_ceaps_limpo.ID.duplicated(keep=False).sum() == df_ceaps_bruto.ID.duplicated(keep=False).sum()


,coluna,tipo antes,tipo depois,ausentes antes,ausentes depois,valores únicos na base bruta
0,ID,string,string,0,0,382061
1,TIPO_DOCUMENTO,string,string,3144,3144,9
2,ANO,string,Int64,0,0,18
3,MÊS,string,Int64,0,0,12
4,COD_SENADOR,string,string,0,0,326
5,NOME_SENADOR,string,string,0,0,326
6,TIPO_DESPESA,string,string,284,284,7
7,CPF_CNPJ_FORNECEDOR,string,string,3144,3144,30734
8,NOME_FORNECEDOR,string,string,3144,3144,59795
9,DOCUMENTO,string,string,27534,27534,261811


,coluna derivada,tipo,ausentes
0,NOME_SENADOR_CHAVE,string,0
1,NOME_FORNECEDOR_CHAVE,string,3144
2,TIPO_DESPESA_CHAVE,string,284
3,NOME_SENADOR_PADRONIZADO,string,0
4,TIPO_DESPESA_PADRONIZADO,string,284
5,DATA_COMPROVANTE,datetime64[us],3144


,métrica,antes,depois
0,registros,382061,382061
1,IDs em repetição,0,0
2,linhas originais idênticas adicionais,0,0


,campo,registros alterados
0,NOME_SENADOR,0
1,NOME_FORNECEDOR,8891
2,TIPO_DESPESA,29827
3,CPF_CNPJ_FORNECEDOR,5
4,rótulo de categoria padronizado por grafia exi...,0
5,nome de senador de exibição por COD_SENADOR,0


Total antes: R$ 422,627,169.14; depois: R$ 422,627,169.14; diferença: R$ 0.00.


In [12]:
df_ceaps_limpo.head(2)

,ID,TIPO_DOCUMENTO,ANO,MÊS,COD_SENADOR,NOME_SENADOR,TIPO_DESPESA,CPF_CNPJ_FORNECEDOR,NOME_FORNECEDOR,DOCUMENTO,DATA,DETALHAMENTO,VALOR_REEMBOLSADO,ANO_ARQUIVO,NOME_SENADOR_CHAVE,NOME_FORNECEDOR_CHAVE,TIPO_DESPESA_CHAVE,NOME_SENADOR_PADRONIZADO,TIPO_DESPESA_PADRONIZADO,DATA_COMPROVANTE
0,2008050351004,<NA>,2008,5,3399,DEMÓSTENES TORRES,"Contratação de consultorias, assessorias, pesq...",<NA>,<NA>,<NA>,<NA>,<NA>,10000.0,2008,demostenes torres,<NA>,"contratacao de consultorias, assessorias, pesq...",DEMÓSTENES TORRES,"Contratação de consultorias, assessorias, pesq...",NaT
1,2008050351003,<NA>,2008,5,3399,DEMÓSTENES TORRES,"Locomoção, hospedagem, alimentação, combustíve...",<NA>,<NA>,<NA>,<NA>,<NA>,3382.37,2008,demostenes torres,<NA>,"locomocao, hospedagem, alimentacao, combustive...",DEMÓSTENES TORRES,"Locomoção, hospedagem, alimentação, combustíve...",NaT


### Base preparada

`df_ceaps_limpo` contém os registros prontos para a análise. `ANO`, `MÊS` e `VALOR_REEMBOLSADO` são numéricos. `COD_SENADOR` e `ID` continuam como texto. As colunas terminadas em `_PADRONIZADO` ou `_CHAVE` ajudam a trabalhar com nomes e categorias. O DataFrame `diagnostico` lista pontos que precisam de revisão. Valores negativos, outliers e ausentes continuam na base.

## Normalização

A [API legislativa do Senado](https://legis.senado.leg.br/dadosabertos/api-docs/swagger-ui/index.html#/) oferece os endpoints `/dadosabertos/senador/{codigo}`, `/filiacoes` e `/mandatos` em JSON (`.json`). A coleta usa os códigos da CEAPS e confere cada resposta com `CodigoParlamentar` no detalhe e `Codigo` nos serviços históricos.

Execute `python coletar_senadores.py` na pasta do projeto antes desta seção. O script consulta os códigos distintos da CEAPS de 2008 a 2025, reutiliza JSONs válidos em `dados_senadores/` e registra as falhas em `dados_senadores/resumo_coleta.json`. Pode ser executado novamente após uma interrupção.

**Formato de entrada:** cada código precisa dos três JSONs originais: `detalhe_{codigo}.json`, `mandatos_{codigo}.json` e `filiacoes_{codigo}.json`. O código no nome do arquivo deve coincidir com `CodigoParlamentar` ou `Codigo` na resposta. A rotina lê apenas os arquivos locais. Para ligar despesas a períodos históricos, precisamos do código parlamentar, da UF e das datas dos mandatos, além do código e da sigla do partido e das datas de filiação e desfiliação. A rotina informa arquivos ausentes e códigos diferentes, sem criar associações por suposição.

In [13]:
import json

def como_lista(valor):
    if valor is None:
        return []
    return valor if isinstance(valor, list) else [valor]

def normalizar_senador(detalhe, mandatos_json, filiacoes_json):
    identificacao = detalhe['DetalheParlamentar']['Parlamentar']['IdentificacaoParlamentar']
    codigo = str(identificacao['CodigoParlamentar'])
    dados_mandatos = mandatos_json['MandatoParlamentar']['Parlamentar']
    dados_filiacoes = filiacoes_json['FiliacaoParlamentar']['Parlamentar']
    if str(dados_mandatos['Codigo']) != codigo or str(dados_filiacoes['Codigo']) != codigo:
        raise ValueError(f'Código parlamentar divergente nos JSONs de {codigo}')

    parlamentar = {'COD_SENADOR': codigo, 'NOME_PARLAMENTAR': identificacao.get('NomeParlamentar'),
                   'NOME_COMPLETO': identificacao.get('NomeCompletoParlamentar')}
    mandatos = []
    exercicios = []
    for mandato in como_lista((dados_mandatos.get('Mandatos') or {}).get('Mandato')):
        primeira = mandato.get('PrimeiraLegislaturaDoMandato') or {}
        segunda = mandato.get('SegundaLegislaturaDoMandato') or {}
        inicios = [x for x in [primeira.get('DataInicio'), segunda.get('DataInicio')] if x]
        fins = [x for x in [primeira.get('DataFim'), segunda.get('DataFim')] if x]
        codigo_mandato = mandato.get('CodigoMandato')
        mandatos.append({'COD_SENADOR': codigo, 'COD_MANDATO': codigo_mandato,
                         'UF_MANDATO': mandato.get('UfParlamentar'),
                         'PARTICIPACAO': mandato.get('DescricaoParticipacao'),
                         'INICIO_MANDATO': min(inicios) if inicios else None,
                         'FIM_MANDATO': max(fins) if fins else None})
        for exercicio in como_lista((mandato.get('Exercicios') or {}).get('Exercicio')):
            exercicios.append({'COD_SENADOR': codigo, 'COD_MANDATO': codigo_mandato,
                               'COD_EXERCICIO': exercicio.get('CodigoExercicio'),
                               'INICIO_EXERCICIO': exercicio.get('DataInicio'),
                               'FIM_EXERCICIO': exercicio.get('DataFim')})

    filiacoes = []
    for filiacao in como_lista((dados_filiacoes.get('Filiacoes') or {}).get('Filiacao')):
        partido = filiacao.get('Partido') or {}
        filiacoes.append({'COD_SENADOR': codigo, 'COD_PARTIDO': partido.get('CodigoPartido'),
                          'SIGLA_PARTIDO': partido.get('SiglaPartido'),
                          'NOME_PARTIDO': partido.get('NomePartido'),
                          'INICIO_FILIACAO': filiacao.get('DataFiliacao'),
                          'FIM_FILIACAO': filiacao.get('DataDesfiliacao')})
    return parlamentar, mandatos, exercicios, filiacoes


In [14]:
PASTA_SENADORES = Path('dados_senadores')
parlamentares, mandatos, exercicios, filiacoes = [], [], [], []
problemas_integracao = []
arquivos_detalhe = sorted(PASTA_SENADORES.glob('detalhe_*.json')) if PASTA_SENADORES.exists() else []
codigos_arquivos = set()
for prefixo in ['detalhe', 'mandatos', 'filiacoes']:
    codigos_arquivos.update(p.stem.removeprefix(prefixo + '_') for p in PASTA_SENADORES.glob(prefixo + '_*.json'))

for codigo_arquivo in sorted(codigos_arquivos):
    arquivo = PASTA_SENADORES / f'detalhe_{codigo_arquivo}.json'
    arquivo_mandatos = PASTA_SENADORES / f'mandatos_{codigo_arquivo}.json'
    arquivo_filiacoes = PASTA_SENADORES / f'filiacoes_{codigo_arquivo}.json'
    ausentes = [p.name for p in [arquivo, arquivo_mandatos, arquivo_filiacoes] if not p.is_file()]
    if ausentes:
        problemas_integracao.append({'codigo_arquivo': codigo_arquivo, 'problema': f'arquivos ausentes: {ausentes}'})
        continue
    try:
        detalhe = json.loads(arquivo.read_text(encoding='utf-8-sig'))
        mandato = json.loads(arquivo_mandatos.read_text(encoding='utf-8-sig'))
        filiacao = json.loads(arquivo_filiacoes.read_text(encoding='utf-8-sig'))
        p, m, e, f = normalizar_senador(detalhe, mandato, filiacao)
        if p['COD_SENADOR'] != codigo_arquivo:
            raise ValueError('código no nome do arquivo difere da resposta')
        parlamentares.append(p)
        mandatos.extend(m)
        exercicios.extend(e)
        filiacoes.extend(f)
    except (OSError, ValueError, KeyError, TypeError) as erro:
        problemas_integracao.append({'codigo_arquivo': codigo_arquivo, 'problema': str(erro)})

df_parlamentares = pd.DataFrame(parlamentares, columns=['COD_SENADOR', 'NOME_PARLAMENTAR', 'NOME_COMPLETO'])
df_mandatos = pd.DataFrame(mandatos, columns=['COD_SENADOR', 'COD_MANDATO', 'UF_MANDATO', 'PARTICIPACAO', 'INICIO_MANDATO', 'FIM_MANDATO'])
df_exercicios = pd.DataFrame(exercicios, columns=['COD_SENADOR', 'COD_MANDATO', 'COD_EXERCICIO', 'INICIO_EXERCICIO', 'FIM_EXERCICIO'])
df_filiacoes = pd.DataFrame(filiacoes, columns=['COD_SENADOR', 'COD_PARTIDO', 'SIGLA_PARTIDO', 'NOME_PARTIDO', 'INICIO_FILIACAO', 'FIM_FILIACAO'])
for tabela, colunas in [(df_mandatos, ['INICIO_MANDATO', 'FIM_MANDATO']),
                        (df_exercicios, ['INICIO_EXERCICIO', 'FIM_EXERCICIO']),
                        (df_filiacoes, ['INICIO_FILIACAO', 'FIM_FILIACAO'])]:
    for coluna in colunas:
        tabela[coluna] = pd.to_datetime(tabela[coluna], format='%Y-%m-%d', errors='coerce')

problemas_integracao = pd.DataFrame(problemas_integracao, columns=['codigo_arquivo', 'problema'])
codigos_ceaps = set(df_ceaps_limpo.COD_SENADOR.dropna())
codigos_normalizados = set(df_parlamentares.COD_SENADOR.dropna())
print(f'Arquivos de detalhe recebidos: {len(arquivos_detalhe)}; normalizados: {len(df_parlamentares)}.')
print(f'Códigos CEAPS: {len(codigos_ceaps)}; com resposta normalizada: {len(codigos_ceaps & codigos_normalizados)}.')
display(problemas_integracao)
qualidade_historica = pd.DataFrame([
    ['mandatos sem início válido', int(df_mandatos.INICIO_MANDATO.isna().sum())],
    ['mandatos com fim anterior ao início', int((df_mandatos.FIM_MANDATO < df_mandatos.INICIO_MANDATO).sum())],
    ['filiações sem início válido', int(df_filiacoes.INICIO_FILIACAO.isna().sum())],
    ['filiações com fim anterior ao início', int((df_filiacoes.FIM_FILIACAO < df_filiacoes.INICIO_FILIACAO).sum())],
], columns=['verificação', 'linhas'])
display(qualidade_historica)
display(pd.DataFrame({'tabela': ['parlamentares', 'mandatos', 'exercícios', 'filiações'],
                      'linhas': [len(df_parlamentares), len(df_mandatos), len(df_exercicios), len(df_filiacoes)]}))


Arquivos de detalhe recebidos: 326; normalizados: 326.
Códigos CEAPS: 326; com resposta normalizada: 326.


,codigo_arquivo,problema


,verificação,linhas
0,mandatos sem início válido,0
1,mandatos com fim anterior ao início,0
2,filiações sem início válido,0
3,filiações com fim anterior ao início,0


,tabela,linhas
0,parlamentares,326
1,mandatos,441
2,exercícios,632
3,filiações,710


### Limites da integração histórica

A normalização usa apenas os JSONs válidos disponíveis em `dados_senadores/`. A cobertura e as falhas da coleta ficam em `resumo_coleta.json` e devem ser conferidas antes de usar as tabelas históricas. Alguns documentos de fornecedores na CEAPS vêm parcialmente mascarados. Não tentamos recuperar os dígitos ocultos. Também não usamos o partido ou a UF atual para completar despesas antigas. A ligação entre as bases deve usar `COD_SENADOR` e os períodos dos mandatos e das filiações, quando forem confirmados. Se houver troca de partido durante um mês e a data do comprovante não permitir identificar a filiação correta, o mês ficará sem partido. Códigos ausentes, períodos sobrepostos e datas inválidas precisam de revisão.

### Integração histórica
Para ampliar as possibilidades de análise, foi realizada uma integração das despesas do CEAPS às informações históricas dos senadores. A integração busca associar cada despesa à unidade federativa e ao partido correspondente ao período de referência adotado

`UF_HISTORICA`: UF associada ao exercício parlamentar na data da despesa

`PARTIDO_HISTORICO`: sigla do partido ao qual o senador estava filiado naquela data

`NOME_PARTIDO_HISTORICO`: nome do partido, quando disponível

In [15]:
despesas = df_ceaps_limpo.copy()

print("Total de despesas:", len(despesas))
print("IDs ausentes:", despesas["ID"].isna().sum())
print("IDs duplicados:", despesas["ID"].duplicated().sum())

despesas["DATA_REF"] = pd.to_datetime(despesas["DATA_COMPROVANTE"], errors="coerce")
tabelas = [ despesas,df_mandatos, df_exercicios,df_filiacoes]
for df in tabelas:df["COD_SENADOR"] = pd.to_numeric(df["COD_SENADOR"],errors="coerce").astype("Int64")
for df in [df_mandatos, df_exercicios]:df["COD_MANDATO"] = pd.to_numeric(df["COD_MANDATO"],errors="coerce").astype("Int64")
for coluna in ["INICIO_MANDATO", "FIM_MANDATO"]:df_mandatos[coluna] = pd.to_datetime(df_mandatos[coluna],errors="coerce")
for coluna in ["INICIO_EXERCICIO", "FIM_EXERCICIO"]:df_exercicios[coluna] = pd.to_datetime(df_exercicios[coluna],errors="coerce")
for coluna in ["INICIO_FILIACAO", "FIM_FILIACAO"]:df_filiacoes[coluna] = pd.to_datetime(df_filiacoes[coluna],errors="coerce")

Total de despesas: 382061
IDs ausentes: 0
IDs duplicados: 0


In [17]:
mandatos_uf = df_mandatos[["COD_SENADOR", "COD_MANDATO", "UF_MANDATO"]].drop_duplicates()

exercicios_com_uf = df_exercicios.merge(mandatos_uf,on=["COD_SENADOR", "COD_MANDATO"],how="left",validate="many_to_one")
candidatos_exercicio = despesas[ ["ID", "COD_SENADOR", "DATA_REF"]].merge(
    exercicios_com_uf[
        ["COD_SENADOR",
            "COD_MANDATO",
            "INICIO_EXERCICIO",
            "FIM_EXERCICIO",
            "UF_MANDATO"
        ]],on="COD_SENADOR",how="left")

condicao_exercicio = (
    candidatos_exercicio["DATA_REF"].notna()
    & candidatos_exercicio["INICIO_EXERCICIO"].notna()
    & (candidatos_exercicio["DATA_REF"]>= candidatos_exercicio["INICIO_EXERCICIO"])
    & (candidatos_exercicio["FIM_EXERCICIO"].isna() | (candidatos_exercicio["DATA_REF"]<= candidatos_exercicio["FIM_EXERCICIO"])))

exercicios_validos = candidatos_exercicio[condicao_exercicio].copy()
quantidade_exercicios = exercicios_validos.groupby("ID").size()
exercicios_unicos = exercicios_validos[exercicios_validos["ID"].map( quantidade_exercicios).eq(1)]
uf_por_despesa = exercicios_unicos.set_index("ID")["UF_MANDATO"]

candidatos_filiacao = despesas[
    ["ID", "COD_SENADOR", "DATA_REF"]].merge(df_filiacoes[
        ["COD_SENADOR",
            "SIGLA_PARTIDO",
            "NOME_PARTIDO",
            "INICIO_FILIACAO",
            "FIM_FILIACAO"]],on="COD_SENADOR",how="left")

condicao_filiacao = (
    candidatos_filiacao["DATA_REF"].notna()
    & candidatos_filiacao["INICIO_FILIACAO"].notna()
    & (candidatos_filiacao["DATA_REF"]>= candidatos_filiacao["INICIO_FILIACAO"])
    & (candidatos_filiacao["FIM_FILIACAO"].isna()| (candidatos_filiacao["DATA_REF"]<= candidatos_filiacao["FIM_FILIACAO"])))

filiacoes_validas = candidatos_filiacao[condicao_filiacao].copy()
quantidade_filiacoes = filiacoes_validas.groupby("ID").size()
filiacoes_unicas = filiacoes_validas[filiacoes_validas["ID"].map(quantidade_filiacoes).eq(1)]
partido_por_despesa = filiacoes_unicas.set_index("ID")["SIGLA_PARTIDO"]
nome_partido_por_despesa = filiacoes_unicas.set_index("ID")["NOME_PARTIDO"]
despesas["PARTIDO_HISTORICO"] = despesas["ID"].map(partido_por_despesa)
despesas["NOME_PARTIDO_HISTORICO"] = despesas["ID"].map(nome_partido_por_despesa)
despesas["UF_HISTORICA"] = despesas["ID"].map(uf_por_despesa)

In [18]:
print("Despesas originais:", len(df_ceaps_limpo))
print("Despesas após integração:", len(despesas))
print("Despesas sem UF histórica:",despesas["UF_HISTORICA"].isna().sum())
print("Despesas sem partido histórico:",despesas["PARTIDO_HISTORICO"].isna().sum())
print("Despesas com mais de um exercício correspondente:",(quantidade_exercicios > 1).sum())
print( "Despesas com mais de uma filiação correspondente:",(quantidade_filiacoes > 1).sum())
assert len(despesas) == len(df_ceaps_limpo)
assert despesas["ID"].equals(df_ceaps_limpo["ID"])

display(despesas[[
            "ID",
            "COD_SENADOR",
            "NOME_SENADOR",
            "DATA_COMPROVANTE",
            "VALOR_REEMBOLSADO",
            "UF_HISTORICA",
            "NOME_PARTIDO_HISTORICO",
            "PARTIDO_HISTORICO"] ].tail(10)
)

Despesas originais: 382061
Despesas após integração: 382061
Despesas sem UF histórica: 5884
Despesas sem partido histórico: 3349
Despesas com mais de um exercício correspondente: 1
Despesas com mais de uma filiação correspondente: 3


,ID,COD_SENADOR,NOME_SENADOR,DATA_COMPROVANTE,VALOR_REEMBOLSADO,UF_HISTORICA,NOME_PARTIDO_HISTORICO,PARTIDO_HISTORICO
382051,2258619,825,PAULO PAIM,2025-05-07,97.89,RS,Partido dos Trabalhadores,PT
382052,2258527,825,PAULO PAIM,2025-05-29,194.71,RS,Partido dos Trabalhadores,PT
382053,2258573,825,PAULO PAIM,2025-05-26,100.0,RS,Partido dos Trabalhadores,PT
382054,2258533,825,PAULO PAIM,2025-05-27,50.0,RS,Partido dos Trabalhadores,PT
382055,2257606,825,PAULO PAIM,2025-05-02,50.0,RS,Partido dos Trabalhadores,PT
382056,2257625,825,PAULO PAIM,2025-05-06,250.0,RS,Partido dos Trabalhadores,PT
382057,2258592,825,PAULO PAIM,2025-05-17,150.0,RS,Partido dos Trabalhadores,PT
382058,2258579,825,PAULO PAIM,2025-05-19,181.06,RS,Partido dos Trabalhadores,PT
382059,2258526,825,PAULO PAIM,2025-05-30,193.66,RS,Partido dos Trabalhadores,PT
382060,2258536,825,PAULO PAIM,2025-05-22,150.0,RS,Partido dos Trabalhadores,PT


In [20]:
despesas.head(3)

,ID,TIPO_DOCUMENTO,ANO,MÊS,COD_SENADOR,NOME_SENADOR,TIPO_DESPESA,CPF_CNPJ_FORNECEDOR,NOME_FORNECEDOR,DOCUMENTO,...,NOME_SENADOR_CHAVE,NOME_FORNECEDOR_CHAVE,TIPO_DESPESA_CHAVE,NOME_SENADOR_PADRONIZADO,TIPO_DESPESA_PADRONIZADO,DATA_COMPROVANTE,DATA_REF,PARTIDO_HISTORICO,NOME_PARTIDO_HISTORICO,UF_HISTORICA
0,2008050351004,<NA>,2008,5,3399,DEMÓSTENES TORRES,"Contratação de consultorias, assessorias, pesq...",<NA>,<NA>,<NA>,...,demostenes torres,<NA>,"contratacao de consultorias, assessorias, pesq...",DEMÓSTENES TORRES,"Contratação de consultorias, assessorias, pesq...",NaT,NaT,NaN,NaN,NaN
1,2008050351003,<NA>,2008,5,3399,DEMÓSTENES TORRES,"Locomoção, hospedagem, alimentação, combustíve...",<NA>,<NA>,<NA>,...,demostenes torres,<NA>,"locomocao, hospedagem, alimentacao, combustive...",DEMÓSTENES TORRES,"Locomoção, hospedagem, alimentação, combustíve...",NaT,NaT,NaN,NaN,NaN
2,2008060351001,<NA>,2008,6,3399,DEMÓSTENES TORRES,"Aluguel de imóveis para escritório político, c...",<NA>,<NA>,<NA>,...,demostenes torres,<NA>,"aluguel de imoveis para escritorio politico, c...",DEMÓSTENES TORRES,"Aluguel de imóveis para escritório político, c...",NaT,NaT,NaN,NaN,NaN


A integração preservou os 382.061 registros de despesas. Após o tratamento das datas, foram identificadas 5.884 despesas sem correspondência histórica de UF e 3.349 sem correspondência de partido. Também foram encontrados registros com mais de uma correspondência possível, assim, as análises por unidade federativa e partido político deverão levar em conta a cobertura das informações históricas e os casos sem correspondência ou com associação ambígua, evitando atribuições indevidas.

### Planejamento da análise
1. Evolução temporal das despesas

    Investigar como os gastos parlamentares variam ao longo dos anos e meses indentificando períodos de aumento ou redução e possíveis padrões temporais. Possível vizualização:
* gráfico de linhas com valor total das depesas por ano
* gráfico de linhas com a evolução mensal dos gastos ao longo do período analisado

2. Distribuição das despesas por categoria

    Analisar quais tipos de despesas apresentam maior participação nos gastos parlamentares identificando as categorias mais frequentes e as que concentram os maiores valores reembolsados. Possível vizualização:
* gráfico de barras com o valor total reembolsado por tipo de despesa 
* gráfico de barras com a quantidade de registros por categori
* comparação entre a participação percentual de cada categoria no total de gastos.

3. distribuição dos gastos entre os senadores

    Identificando aqueles com maiores e menores gastos e verificando se existem diferenças expressivas entre os parlamentares. Comparar os gatos entre sendores que exerceram distintos tempos de mandatos torna a analise eveiesada, por isso, além do valor total vale considerar o gasto médio por mês de exercício, quando for possível.
* gráfico de barras com os senadores que apresentam os maiores valores totais de despesas.
* comparação entre a quantidade de despesas e o valor total reembolsado por senador.
* análise da distribuição dos valores reembolsados, considerando medidas como média, mediana e dispersão

4. Comparação dos gastos por UF

    Utilizar os dados históricos dos mandatos para investigar a distribuição dos gastos entre as unidades federativas dos senadores.
* gráfico de barras com o valor total reembolsado por UF.
* comparação entre a quantidade de despesas e o valor total reembolsado por estado
* evolução temporal dos gastos por UF

5. Comparação dos gastos por partido político

    Utilizar o histórico de filiações partidárias para investigar como os gastos se distribuem entre os partidos aos quais os senadores estavam vinculados no período analisado, tem que considerar as quantidades diferentes de senadores por partido, porque gasto do partido total não significa maior gasto individualmente e partidos com mais senadores podem acumular despesas maiores, assim, a análise do gasto médio por senador ou mês de exercício se torna mais informativa.
* gráfico de barras com o valor total reembolsado por partido
* comparação entre a quantidade de despesas por partido
* evolução dos gastos por partido ao longo do tempo

6. análise dos reembolsos

    Investigar a distribuição dos valores das despesas individuais, identificando sua concentração, dispersão e possíveis outliers
* histograma dos valores reembolsados
* Boxplot dos valores das despesas
* comparação entre a média, modo e mediana?

